In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import re
import pymupdf
import pymupdf4llm
import pathlib as pl
import itertools
from IPython.display import display, Markdown

from dotenv import load_dotenv
# noinspection trailing-semicolon
load_dotenv();  # Suppress output

from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter, MarkdownTextSplitter
from langchain_core.documents import Document
from langchain_core.runnables import RunnableConfig

from tqdm.auto import tqdm

from risk_pipeline.models import ExtractionResult
from IPython.core.magic import register_cell_magic

In [3]:
@register_cell_magic
def skip(line, cell):
    message = f"skipping: {line.strip()}" if line.strip() else "skipping"
    print(message)

needed_env = ["OPENAI_API_KEY"]
for e in needed_env: assert os.getenv(e), f"{e} is not set. Please check your .env file or environment variables."
display(Markdown('**<span style="color:green">Ready to go</span>**'))

**<span style="color:green">Ready to go</span>**

In [4]:
case_path = pl.Path.cwd() / "doc" / "Case_GenAIandNLPEngineer.pdf"
md_case = pymupdf4llm.to_markdown(case_path)
print(md_case)

# Case Study: Building a Structured Risk Intelligence pipeline 

The following case study is intended to measure your ability to understand, analyse, solve and advise on a business problem. We will use it to assess how you approach the problem, propose and construct a reasonable solution and layout a strategy for deployment in production including potential future improvements. We are interested in discussing the proposed solution at a high-level from a stakeholder engagement point of view, as well as at a deep technical level. 

As time for completing the task is very limited (~4–8 hours), we suggest that all attempts (even unsuccessful ones) are kept for discussion. We are interested in your train-of-thought process, how that is put into use in your technical solution and how you think about building a real product, more than in the level of performance of the solution itself. 

We expect you to use AI coding assistants (Cursor, Copilot, Claude Code, your IDE's built-in assistant, wh

# Parsing and extraction

In [5]:
report_path = pl.Path.cwd() / "data" / "VestasAnnualReport2025.pdf"
pages_of_interest = ((range(49, 51), "Risk Management"), (range(70, 74), "Material Risk"), (range(84, 92), "Climate Change"),(range(117, 118), "Cyber Security")) # Lesson learned - keep these in order, otherwise markdown extraction and secton reconstruction does not match

In [10]:
doc_src = pymupdf.open(report_path)

In [6]:
%%skip No need to show this anymore
doc_src[117].get_pixmap().pil_image()

skipping: No need to show this anymore


### Copy relevant pages from pdf

In [8]:
%%skip Only necessary once
doc_dst = pymupdf.open()

for (page_range, topic) in pages_of_interest:
    for page_num in page_range:
        page_src = doc_src[page_num]
        page_dst = doc_dst.new_page(-1, page_src.rect.width, page_src.rect.height)
        page_dst.show_pdf_page(page_src.rect, doc_src, page_num)

pp = pl.Path(doc_src.name).with_stem(pl.Path(doc_src.name).stem + "-extract")

doc_dst.save(pp,
    garbage=3,  # eliminate duplicate objects
    deflate=True,  # compress stuff where possible
)

skipping: Only necessary once


### Load all pages of interest into markdown

In [11]:
pages = [page for (page_range, _) in pages_of_interest for page in page_range]
md = pymupdf4llm.to_markdown(doc_src, pages=pages, header=False, footer=False, page_chunks=True, show_progress=True)

# Reconstruct sections
it = iter(md)
md_sections = [
    (section, page_range, list(itertools.islice(it, len(page_range))))
    for page_range, section in pages_of_interest
]

Parsing 15 pages of '/Users/timkaas/Projects/vestas-risk-case/data/VestasAnnualReport2025.pdf'...


100%|██████████| 15/15 [00:02<00:00,  5.67it/s]


Generating markdown text...


100%|██████████| 15/15 [00:00<00:00, 5734.10it/s]


### Document splitting

Let’s test if splitting the document into chunks yields a higher recall to prevent a "lost-in-the-middle" effect. \
Potential pitfalls:
- Since the reports are converted from PDF to Markdown, the splits might be messed up due to incorrect headline parsing
- Splitting could lead to a loss of context if not split properly
- Splitting could carry a higher cost in terms of tokenization due to repeated prompt overhead.
- The same risk can be reported twice if split in context

Benefits:
- A split document can be ingested in parallel, speeding up the processing. However, I do not think speed is the main concern here

In [ ]:
splitter = MarkdownTextSplitter(chunk_size=400, chunk_overlap=20)
md_splits = splitter.create_documents([md[0]['text']])

md_splits
#md_header_splits = markdown_splitter.split_text(markdown_document)

In [13]:
def format_pages(pages) -> str:
    return "\n\n".join(
        f"=== PDF PAGE {page['metadata']['page_number']} ===\n\n{page['text']}"
        for page in pages
    )

def format_sections(sections):
    return "\n\n".join(f"# {section_name}\n\n{format_pages(pages)}" for section_name, page_rang, pages in sections)

In [ ]:
display(Markdown(format_sections(md_sections)))

In [137]:
headers_to_split_on = [
    ("#", "Header 1"),
    #("##", "Header 2"),
    #("###", "Header 3"),
]

#full_md = format_pages(md)
full_md = format_sections(md_sections)

#mdd = Document(page_content=md[0]['text'], metadata=md[0]['metadata'])

splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
#md_splits = sum((splitter.split_text(mdp['text']) for mdp in md), [])
md_splits = splitter.split_text(full_md)

current_page = []
# 3. Extract page numbers into metadata for each split
for doc in md_splits:
    # Find all page markers contained within this chunk
    found_pages = [int(p) for p in re.findall(r"=== PDF PAGE (\d+) ===", doc.page_content)]

    if found_pages:
        current_page = found_pages

    # Store page metadata
    doc.metadata["pages"] = current_page

In [139]:
[m.metadata for m in md_splits]
md_splits[0]

Document(metadata={'Header 1': 'Risk Management', 'pages': [50]}, page_content='# Risk Management  \n=== PDF PAGE 50 ===')

# Datamodels

In [60]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

EXTRACTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are an expert risk intelligence analyst. Extract all principal corporate risks from the provided annual report text with high precision."),
    ("human", "Section: {section_name}\nPages: {page_numbers}\n\nContent:\n{content}")
])

def build_extraction_chain(model_name: str = "gpt-4o", temperature: float = 0.0):
    llm = ChatOpenAI(model=model_name, temperature=temperature)
    structured_llm = llm.with_structured_output(ExtractionResult)
    return EXTRACTION_PROMPT | structured_llm

In [170]:
%%skip Test call to LLM
# Test invokation of a single section
llm_chain = build_extraction_chain()
section_name, page_range, pages = md_sections[0]
result = llm_chain.invoke({"section_name": section_name, "page_numbers": list(page_range), "content": format_pages(pages)})

skipping: Test call to LLM


In [148]:
batch_inputs = [
    {
        "section_name": section_name,
        "page_numbers": list(page_range),
        "content": format_pages(pages),
    }
    for section_name, page_range, pages in md_sections
]

results = llm_chain.batch(batch_inputs, config=RunnableConfig(max_concurrency=4))

In [156]:
#ExtractionResult(**(r.model_dump() for r in results))
risks = [rr.model_dump() for r in results for rr in r.risks]
result = ExtractionResult(risks=risks)

In [174]:
for r in risks:
    print(r)

{'title': 'Geopolitical and Regulatory Framework', 'description': 'Vestas faces significant challenges due to shifting geopolitical tensions and regulatory changes. Conflicts in regions like Ukraine and the Middle East disrupt stability, affecting global supply chains and operations. Trade tensions and assertive industrial policies in major economies further complicate the regulatory landscape, impacting market incentives and competition in the clean-tech sector.', 'category': <RiskCategory.GEOPOLITICAL: 'geopolitical'>, 'section': 'Main risks', 'pages': [51], 'mitigation': 'Vestas manages geopolitical risks by monitoring global developments, maintaining a global and regional manufacturing footprint, and implementing appropriate mitigations.', 'evidence': [{'page': 51, 'quote': 'In 2025, Vestas continued to navigate significant challenges arising from shifting geopolitical tensions across the world. Conflicts in Ukraine and the Middle East continued to disrupt regional stability, affec

In [175]:
from risk_pipeline.parser import parse_page_ranges

parse_page_ranges("1-6,8,12-14")

[SectionSpec(name='Pages 1-6', page_range=[0, 1, 2, 3, 4, 5], description='Extracted from report pages 1-6'),
 SectionSpec(name='Page 8', page_range=[7], description='Extracted from report page 8'),
 SectionSpec(name='Pages 12-14', page_range=[11, 12, 13], description='Extracted from report pages 12-14')]

In [194]:
from typing import Iterable
from src import RiskCategory
from dataclasses import dataclass
import json

@dataclass
class GoldenData:
    id: str
    section: str
    pages: List[int]
    category: RiskCategory
    title_keywords: List[str]
    expected_mitigation: bool
    key_mitigation_keywords: List[str]
    reference_quote_snippet: str


def load_from_jsonl(file_path) -> Iterable[GoldenData]:
    with open(file_path, 'r') as file:
        reader = json.load(file)
        return [GoldenData(**data) for data in reader]

In [195]:
load_from_jsonl(pl.Path.cwd() / "src/eval/golden.json")

[GoldenData(id='risk_geo_01', section='Risk Management', pages=[50, 51], category='geopolitical', title_keywords=['geopolitical', 'regulatory framework', 'trade tensions'], expected_mitigation=True, key_mitigation_keywords=['scenario planning', 'supply chain diversification', 'advocacy'], reference_quote_snippet='Conflicts in regions like Ukraine and the Middle East'),
 GoldenData(id='risk_cyber_01', section='Cyber Security', pages=[118], category='cyber', title_keywords=['cyber attack', 'information security', 'ransomware'], expected_mitigation=True, key_mitigation_keywords=['24/7 monitoring', 'zero trust', 'SOC'], reference_quote_snippet='cyber security incidents')]

# Sentence segmentation

In [17]:
import spacy
from typing import List

nlp = spacy.load("en_core_web_sm")

# Risk-signal vocabulary — could be expanded significantly
RISK_SIGNALS = {
    "risk", "exposure", "threat", "uncertainty", "material impact",
    "principal risk", "vulnerability", "disruption", "adverse",
    "mitigation", "control", "safeguard", "contingency",
}



def filter_risk_sentences(text: str) -> List[str]:
    """
    Segment text into sentences and return only those
    containing at least one risk-signal keyword.
    """
    doc = nlp(text)
    risk_sentences = []

    for sent in doc.sents:
        sent_lower = sent.text.lower()
        if any(signal in sent_lower for signal in RISK_SIGNALS):
            risk_sentences.append(sent.text.strip())

    return risk_sentences

In [19]:
filtered = filter_risk_sentences(format_pages( md_sections[0][2]))

In [20]:
filtered

['=== PDF PAGE 50 ===\n\n# Risk management \n\n##### Enterprise Risk Management annual wheel \n\nRisk Committee Executive management team Board of Directors/Audit committee \n\nQ4 Q1 Focus on persistent risks Review of emerging risks. and mitigations.',
 'Focus on persistent risks and mitigations, incl.',
 'Audit Committee and Board review Q1 + Q2 topics \n\nAt Vestas, risk management is an integral part of business operations and strategy.',
 'We are committed to proactively identifying and mitigating risks that may affect our short- to medium-term objectives, while addressing long-term risks that could hinder the realisation of our strategic goals.',
 '##### Enterprise Risk Management \n\nOperating across diverse markets and a rapidly evolving industry landscape, Vestas is exposed to a wide array of risks that reflect the complexity and global nature of our business.',
 'These risks include operational, commercial, macroeconomic, and regulatory challenges.',
 'Our Enterprise Risk Man